# HPC Diagnostics Notebook

This notebook collects environment, versions, and a minimal repro for the ContinuousIQN eval mismatch.

## 1) Detect Environment and Paths

In [ ]:
import json
import os
import sys
from pathlib import Path

print("cwd:", os.getcwd())
print("python:", sys.executable)
print("sys.path (first 10):")
for p in sys.path[:10]:
    print("  ", p)

env_keys = [
    "HOME", "USER", "SHELL", "PATH", "PYTHONPATH",
    "CONDA_PREFIX", "VIRTUAL_ENV",
]
print("\nselected env vars:")
for k in env_keys:
    print(f"{k}={os.environ.get(k)}")

workspace = Path.cwd()
print("\nworkspace contents (top-level):")
for item in sorted(workspace.iterdir()):
    print("  ", item.name)

## 2) Capture Python and Package Versions

In [ ]:
import platform
from importlib import metadata

print("python version:", platform.python_version())

packages = ["numpy", "pandas", "torch", "click", "yaml", "seaborn", "matplotlib"]
versions = {}
for pkg in packages:
    try:
        versions[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        versions[pkg] = None

print("package versions:")
for k, v in versions.items():
    print(f"  {k}: {v}")

print("torch cuda available:", getattr(__import__("torch"), "cuda").is_available())

## 3) Run Minimal Repro Script

In [ ]:
import traceback
import numpy as np
import yaml
import torch

# EDIT THESE PATHS FOR YOUR HPC RUN
config_name = "iqn_continuous_mimic"  # without .yaml
checkpoint_dir = "runs/iqn_continuous_mimic"
encoded_dir = "data/continuous_mimic/rectilinear_processed"

cfg_path = os.path.join("configs", f"{config_name}.yaml")
print("config:", cfg_path)
print("checkpoint_dir:", checkpoint_dir)
print("encoded_dir:", encoded_dir)

params = yaml.safe_load(open(cfg_path))
params["checkpoint_fname"] = checkpoint_dir

# Load encoded data shape
encoded = np.load(os.path.join(encoded_dir, "encoded_test.npz"), allow_pickle=True)
state_dim = encoded["states"].shape[-1]
action_dim = params.get("action_dim", 2)
print("encoded state_dim:", state_dim)
print("action_dim:", action_dim)

# Load checkpoint head shape
ckpt_path = os.path.join(checkpoint_dir, "best_q_parametersnegative.pt")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
head_w = ckpt["rl_network_state_dict"]["head.weight"]
print("checkpoint head.weight shape:", tuple(head_w.shape))
print("checkpoint inferred state_dim:", head_w.shape[1] - action_dim)

# Attempt load to show any mismatch
from rl_utils import ContinuousIQN_OfflineAgent

try:
    params["input_dim"] = state_dim
    model = ContinuousIQN_OfflineAgent(state_dim, params, sided_Q="negative", device="cpu")
    model.network.load_state_dict(ckpt["rl_network_state_dict"])
    print("load_state_dict: OK")
except Exception as e:
    print("load_state_dict failed:")
    print(repr(e))
    traceback.print_exc()

## 4) Collect System and Scheduler Info

In [ ]:
import platform

print("platform:", platform.platform())
print("processor:", platform.processor())

try:
    import torch
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device count:", torch.cuda.device_count())
        print("cuda device 0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch cuda query failed:", e)

slurm_keys = [k for k in os.environ.keys() if k.startswith("SLURM_")]
print("\nSLURM vars:")
for k in sorted(slurm_keys):
    print(f"{k}={os.environ.get(k)}")

## 5) Save Results to a Report File

In [ ]:
report = {
    "cwd": os.getcwd(),
    "python": sys.executable,
    "sys_path": sys.path[:20],
    "env": {k: os.environ.get(k) for k in ["HOME", "USER", "SHELL", "PATH", "PYTHONPATH", "CONDA_PREFIX", "VIRTUAL_ENV"]},
}

try:
    report["python_version"] = platform.python_version()
except Exception:
    report["python_version"] = None

try:
    report["torch_cuda_available"] = torch.cuda.is_available()
except Exception:
    report["torch_cuda_available"] = None

report_path = "hpc_diagnostics_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("wrote", report_path)